# IPPU BAU uplift — subir procesos industriales de ~7.8 Mt a ~14 Mt (2050)

**Objetivo.** El BAU actual produce **7.8 MtCO2e** en IPPU (subsectores 2.A-2.F, procesos no-energía) en 2050. La SNBC Reference (Figura 28, p.58) muestra **~14 Mt** en 2050 (+80% vs 2021, explícitamente atribuido a la **expansión de fosfato OCP**).

**Diagnóstico.** Tu BAU está dominado por 2.A.1 Cement (~7 Mt). 2.B Chemicals (fosfato/ácido fosfórico OCP) aparece casi invisible cuando debería ser un driver mayor. El gap se llena casi entero subiendo chemicals.

| Driver | Hoy | Problema |
|---|---|---|
| `demscalar_ippu_chemicals` | 1.000 (constante) | Sin step-change para OCP commissioning |
| `demscalar_ippu_cement` | 1.000 (constante) | OK pero falta crecimiento por construcción |
| `demscalar_ippu_metals` | 1.000 (constante) | Infraestructura OCP necesita acero |
| `demscalar_ippu_product_use_ods_refrigeration` | 0.229 (constante) | Penetración AC crece — HFCs deberían subir |

**Estrategia.** Aplicar uplifts diferenciados sobre `demscalar_ippu_*` (sólo tp ≥ 8):
- **Chemicals: 1.0 → 2.00** (rampa por expansión OCP — el driver mayor)
- **Cement: 1.0 → 1.20** (crecimiento moderado con construcción)
- **Metals: 1.0 → 1.30** (infraestructura industrial)
- **HFC refrigeración: 0.229 → 0.50** (multiplicador 2.18x — penetración AC)

**Relación con la libreta INEN.** Esta libreta es **independiente** de `inen_bau_uplift.ipynb`. Ojo:
- Si corres SÓLO IPPU → usa los defaults de arriba.
- Si vas a apilar IPPU sobre INEN (recomendado para BAU final, las elasticidades de chemicals ya están en 0.80 ahí), pon `INPUT_FILE = sisepuede_raw_input_morocco_fuels_inen_bau.csv` y baja chemicals demscalar a **1.60** (porque la elasticidad ya levantó producción).

**Salida.** `sisepuede_raw_input_morocco_fuels_ippu_bau.csv` en `input_data/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR = REPO / 'ssp_modeling' / 'input_data'

# Cambia a 'sisepuede_raw_input_morocco_fuels_inen_bau.csv' para apilar sobre INEN
INPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe_inen.csv'
OUTPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe_inen_ippu.csv'

# --- Knobs ---
TP_CALIB_END = 7
TP_TARGET = 35

# Targets de demscalar en tp=35 (mantienen constante para tp>35)
# Si apilas sobre INEN (elasticidad chemicals=0.80 ya aplicada), baja CHEMICALS a 1.60
TARGETS = {
    'demscalar_ippu_chemicals': 2.00,     # OCP fosfato — driver principal
    'demscalar_ippu_cement': 1.20,        # construcción
    'demscalar_ippu_metals': 1.30,        # infraestructura industrial
    'demscalar_ippu_product_use_ods_refrigeration': 0.50,  # AC penetración
}

pd.options.display.float_format = '{:.4f}'.format
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {INPUT_FILE.name}: {df.shape}  tp=[{df.time_period.min()}, {df.time_period.max()}]')

Loaded sisepuede_raw_input_morocco_fuels_scoe_inen.csv: (56, 2442)  tp=[0, 55]


## 1 · Auditoría — valores actuales

In [2]:
snapshot_tps = [0, 7, 8, 15, 25, 35, 45, 55]
df.loc[df.time_period.isin(snapshot_tps), ['time_period', 'year'] + list(TARGETS.keys())].set_index('time_period')

,year,demscalar_ippu_chemicals,demscalar_ippu_cement,demscalar_ippu_metals,demscalar_ippu_product_use_ods_refrigeration
time_period,,,,,
0,2015,1,1,1,0.2290
7,2022,1,1,1,0.2290
8,2023,1,1,1,0.2290
15,2030,1,1,1,0.2290
25,2040,1,1,1,0.2290
35,2050,1,1,1,0.2290
45,2060,1,1,1,0.2290
55,2070,1,1,1,0.2290


## 2 · Construir trayectorias

Cada driver rampa linealmente desde su valor en `TP_CALIB_END` (típicamente 1.0, o 0.229 para HFCs) hasta `TARGETS[col]` en `TP_TARGET`, y se mantiene constante para tp>35.

In [3]:
tp = df['time_period'].to_numpy()
df_new = df.copy()

for col, target in TARGETS.items():
    start_val = df[col].iloc[TP_CALIB_END]  # valor al final de calibración
    traj = df[col].to_numpy().copy()
    ramp = (tp > TP_CALIB_END) & (tp <= TP_TARGET)
    traj[ramp] = np.interp(
        tp[ramp],
        [TP_CALIB_END + 1, TP_TARGET],
        [start_val, target],
    )
    traj[tp > TP_TARGET] = target
    df_new[col] = traj

print('=== After ===')
df_new.loc[df_new.time_period.isin(snapshot_tps), ['time_period', 'year'] + list(TARGETS.keys())].set_index('time_period')

=== After ===


,year,demscalar_ippu_chemicals,demscalar_ippu_cement,demscalar_ippu_metals,demscalar_ippu_product_use_ods_refrigeration
time_period,,,,,
0,2015,1,1,1,0.2290
7,2022,1,1,1,0.2290
8,2023,1,1,1,0.2290
15,2030,1,1,1,0.2993
25,2040,1,1,1,0.3996
35,2050,2,1,1,0.5000
45,2060,2,1,1,0.5000
55,2070,2,1,1,0.5000


## 3 · Verificación — calibración histórica intacta

In [4]:
cols = list(TARGETS.keys())
diff = (df_new[cols] - df[cols]).abs()
calib_max = diff.loc[df['time_period'] <= TP_CALIB_END].max().max()
print(f'Max diff en tp=0..7 (debe ser 0): {calib_max:.2e}')
assert calib_max < 1e-12, 'Calibración tp=0..7 fue modificada — abortar.'
print('OK: calibración intacta.')
print()
print('Cambios en tp=35:')
for c in cols:
    print(f'  {c}: {df[c].iloc[35]:.3f} → {df_new[c].iloc[35]:.3f}  ({df_new[c].iloc[35]/df[c].iloc[35]:.2f}x)')

Max diff en tp=0..7 (debe ser 0): 0.00e+00
OK: calibración intacta.

Cambios en tp=35:
  demscalar_ippu_chemicals: 1.000 → 2.000  (2.00x)
  demscalar_ippu_cement: 1.000 → 1.000  (1.00x)
  demscalar_ippu_metals: 1.000 → 1.000  (1.00x)
  demscalar_ippu_product_use_ods_refrigeration: 0.229 → 0.500  (2.18x)


## 4 · Guardar nuevo CSV

In [5]:
df_new.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {OUTPUT_FILE}')
print(f'Shape: {df_new.shape}')

Wrote /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels_scoe_inen_ippu.csv
Shape: (56, 2442)


## 5 · Próximo paso e iteración

Correr el modelo apuntando al CSV de salida y verificar IPPU en 2050:
- Si **<12 Mt**: subir chemicals → `demscalar_ippu_chemicals: 2.50` y/o cement `1.30`.
- Si **>16 Mt**: bajar chemicals → `demscalar_ippu_chemicals: 1.60`.
- Si **13-15 Mt**: 👍.

**Validación por subsector** (mirar el gráfico Tableau después de correr):
- 2.A.1 Cement: ~7-8 Mt en 2050 (vs 7.0 actual).
- 2.B Chemicals: **debe pasar de ~0.1 → 4-5 Mt** (esto es lo que falta hoy).
- 2.C Metals: ~0.6-0.8 Mt.
- 2.F.1 Refrigeration HFCs: ~0.4-0.6 Mt.

**Nota OCP.** La SNBC implica que la expansión es por *fases* (commissioning en 2027, 2030, 2035). Si quieres ser más fiel al perfil real en lugar de rampa lineal, puedes editar manualmente las filas de `demscalar_ippu_chemicals` con saltos discretos. Empieza con la rampa lineal y refina si la curva no matchea.